# Figures 3, 4, S4

In [ ]:
from datetime import datetime
import pandas as pd
import numpy as np
import os
import time

import plotly.offline as pyo
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

pio.templates.default = "plotly_white" # https://medium.com/plotly/introducing-plotly-py-theming-b644109ac9c7

directory = r'C:\Users\markm\OneDrive\Akadeemiline\PhD RESEARCH\Artiklid\Ukraine-Twitter\Data\Final datasets used in article'
os.chdir(directory)

#for fig rendering errors:
#plotly.offline.init_notebook_mode(connected=True)
#pio.renderers.default = 'iframe_connected' 
#from plotly.offline import init_notebook_mode, iplot
#init_notebook_mode(connected=True)
#pyo.init_notebook_mode()


In [ ]:
#load data
DF = pd.read_csv(r'28languagesUA.csv', sep=',') #, index_col=[0,1], skipinitialspace=True)
DF.date=pd.to_datetime(DF.date)#,dayfirst=True
DF.sample(3)

In [ ]:
#DF2014 = DF[(DF['date'] >= '2014-01-01') & (DF['date'] <= '2014-04-30')]
#DF2022 = DF[(DF['date'] >= '2022-01-01') & (DF['date'] <= '2022-04-30')]

#DF2014 = DF[(DF['date'] >= '2014-01-30') & (DF['date'] <= '2014-03-27')] #Enne valitud
#DF2022 = DF[(DF['date'] >= '2022-01-27') & (DF['date'] <= '2022-03-24')] # Enne valitud

#DF2014 = DF[(DF['date'] >= '2014-02-06') & (DF['date'] <= '2014-03-19')]
#DF2022 = DF[(DF['date'] >= '2022-02-03') & (DF['date'] <= '2022-03-16')]



# Define the central dates as datetime objects
central_dates = {
    '2014': pd.to_datetime('2014-02-27'),
    '2022': pd.to_datetime('2022-02-24')
}

# Filter the DataFrame based on 4 weeks before and after the central dates
DF2014 = DF[(DF['date'] >= central_dates['2014'] - pd.Timedelta(weeks=4)) & 
            (DF['date'] <= central_dates['2014'] + pd.Timedelta(weeks=4))]
#DF2014 = DF[(DF['date'] >= central_dates['2014'] - pd.Timedelta(weeks=1)) & 
#            (DF['date'] <= central_dates['2014'] + pd.Timedelta(weeks=3))]

DF2022 = DF[(DF['date'] >= central_dates['2022'] - pd.Timedelta(weeks=4)) & 
            (DF['date'] <= central_dates['2022'] + pd.Timedelta(weeks=4))]

DF2014.head(3)

DF2014['date'] = DF2014['date'] + pd.DateOffset(years=8, days=-3) #offset 2014 to make em comparable
#DF2014['date'] = DF2014['date'] + pd.DateOffset(years=8, days=6) #offset 2014 to make em comparable 18 and 24

DF2014.head(3)

##### Normalized

In [ ]:
#NORMALIZE BY LOCAL AVG
import numpy as np
import pandas as pd
Overal_mean_freq = DF.groupby('language')['freq'].transform('mean')

def calculate_normalized_frequency(df, freq_col='freq', lang_col='language'):
    mean_freq = df.groupby(lang_col)[freq_col].transform('mean')
    normalized_freq = (df[freq_col] / mean_freq)
    return normalized_freq

def calculate_normalized_overal_frequency(df, freq_col='freq', lang_col='language'):
    normalized_freq = np.log10(df[freq_col] / Overal_mean_freq)
    return normalized_freq

# Calculate overall mean frequency for each language in DF
Overal_mean_freq = DF.groupby('language')['freq'].mean()

def calculate_normalized_overal_frequency(df, overall_mean_freq, freq_col='freq', lang_col='language'):
    # Ensure we only calculate for valid entries
    normalized_freq = (df[freq_col] / overall_mean_freq[df[lang_col]])
    return normalized_freq


# Example usage:
DF2014['normalized_loc_freq'] = calculate_normalized_frequency(DF2014)
DF2022['normalized_loc_freq'] = calculate_normalized_frequency(DF2022)
DF2014['normalized_overal_freq'] = calculate_normalized_frequency(DF2014)
DF2022['normalized_overal_freq'] = calculate_normalized_frequency(DF2022)
DF2014['normalized_overal_freq_log'] = np.log10(calculate_normalized_frequency(DF2014))
DF2022['normalized_overal_freq_log'] = np.log10(calculate_normalized_frequency(DF2022))
DF2022['freq_log'] = np.log10(DF2022['freq'])
DF2014['freq_log'] = np.log10(DF2014['freq'])


DF2022.head(3)

# Plotting 2014 vs 2022 comparison

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd

def plot_frequency_comparison(df_2014, df_2022, date_col='date', freq_col='freq', lang_col='language',
                               title='', 
                               color_2014='#4d99c6', color_2022='#bd2f36'):
    # Get unique languages
    unique_languages = [
        "Indonesian", "Portuguese", "Urdu", "Korean", "Persian", "Spanish", "Turkish", "Arabic", 
        "Catalan", "Danish", "Dutch", "English", "Finnish", "French", "German", "Greek", "Italian", "Norwegian",
        "Polish", "Romanian", "Swedish","Czech","Hungarian",
        "Estonian", "Vietnamese", "Serbian", 
        "Ukrainian", "Russian"
    ]
    unique_languages = [
         'Ukrainian', 'Russian', 'Arabic', 'Portuguese', 'Catalan', 'Korean', 'Persian', 'Turkish', 'Indonesian', 'Urdu', 'Vietnamese',
        'Serbian', 'Estonian', 'Romanian', 'Greek', 'Hungarian', 'Polish', 'Swedish', 'Czech', 'Spanish', 
        'Danish', 'English', 'Dutch', 'Norwegian','Finnish', 'French', 'Italian', 'German'
    ]

    # Calculate number of rows and columns needed for the grid
    num_languages = len(unique_languages)
    num_rows = (num_languages // 4) + 1
    num_cols = min(num_languages, 4)

    # Define specific date ranges for x-axis labels
    feb_24_2022 = pd.to_datetime('2022-02-24')
    date_intervals = [(feb_24_2022 - pd.Timedelta(weeks=i)).strftime('%Y-%m-%d') for i in range(4, 0, -1)]
    date_intervals += ['2022-02-24']
    date_intervals += [(feb_24_2022 + pd.Timedelta(weeks=i)).strftime('%Y-%m-%d') for i in range(1, 5)]
    date_labels = ["4 weeks before", "3 weeks before", "2 weeks before", "1 week before", 
                   "Feb 24", "1 week after", "2 weeks after", "3 weeks after", "4 weeks after"]

    # Create a grid of subplots with shared x and y axes, and reduce vertical spacing
    fig = make_subplots(rows=num_rows, cols=num_cols, subplot_titles=unique_languages, 
                        shared_xaxes=True, shared_yaxes=True, vertical_spacing=0.02, horizontal_spacing=0.02)

    # Iterate through each unique language
    for i, language in enumerate(unique_languages):
        # Filter data for the current language
        df_lang_2022 = df_2022[df_2022[lang_col] == language]
        df_lang_2014 = df_2014[df_2014[lang_col] == language]

        # Calculate subplot row and column
        row = (i // num_cols) + 1
        col = (i % num_cols) + 1
        
        # Add a line plot for 2022
        fig.add_trace(
            go.Scatter(x=df_lang_2022[date_col], y=df_lang_2022[freq_col], mode='lines', 
                       name=language, line=dict(color=color_2022, width=4)),
            row=row, col=col
        )
        
        # Add a line plot for 2014
        fig.add_trace(
            go.Scatter(x=df_lang_2014[date_col], y=df_lang_2014[freq_col], mode='lines', 
                       name=language + ' 2014', line=dict(color=color_2014, width=4)),
            row=row, col=col
        )

        # Add a vertical line on 24th of February
        fig.add_vline(x='2022-02-24', line=dict(color='black', width=2, dash='dot'),
                       row=row, col=col)

    # Define the y-axis range for comparability
    y_axis_range = [1e-7, 1e-2]  # Adjust this range to fit your data best

    # Update layout with reduced margins and increased font size
    fig.update_layout(
        title=title,
        showlegend=False,
        margin=dict(l=10, r=10, t=40, b=10),  # Set left, right, top, and bottom margins
        height=2718,
        width=2127,
        font=dict(size=30)
    )
    
    fig.update_annotations(font=dict(size=32))  # Change all title font sizes

    # Set consistent y-axis range across all subplots and use log scale
    fig.update_yaxes(
        scaleanchor="x", scaleratio=1, type='log', showline=True,
        linewidth=1, linecolor='black', mirror=True,
        tickvals=[1e-6, 1e-5, 1e-4, 1e-3],
        range=[-7, -2]  # Adjusted to match the `y_axis_range`
    )

    # Apply consistent x-axis tick settings across all subplots
    fig.update_xaxes(
        constrain='domain', showline=True, linewidth=1,
        linecolor='black', mirror=True,
        tickvals=date_intervals,
        ticktext=date_labels
    )
    # Make the grid lines darker
    fig.update_xaxes(gridcolor='#cccccc', gridwidth=1.1, row=None, col=None)  # Applies to all x-axes
    fig.update_yaxes(gridcolor='#cccccc', gridwidth=1.1, row=None, col=None)  # Applies to all y-axes


    # Show the plot
    return fig

# Example usage:
fig = plot_frequency_comparison(DF2014, DF2022, date_col='date', freq_col='freq', lang_col='language')
fig.show()


In [ ]:
output_dir= r'C:\Users\markm\OneDrive\Akadeemiline\PhD RESEARCH\Artiklid\Ukraine-Twitter'

In [ ]:
import os
from subprocess import call

# Set the figure directory and filename criteria
sublocation = '/Visualizations/Fig.4_2014vs2022_trends/27022018 and 24022022/'

# Define the plot name
name = '2014vs2022_trends_4w_notnormalized_daily'  # Replace with your desired plot name
#name = 'Wordform_comparison_ADJ'  # Replace with your desired plot name

# Define the base directory for saving files
directory = output_dir
os.chdir(directory)
# Make the main figure directory if not present
main_figure_path = directory + sublocation
os.makedirs(main_figure_path, exist_ok=True)

# Define output formats and create their subdirectories
formats = ['pdf', 'jpeg', 'svg']
for fmt in formats:
    os.makedirs(main_figure_path + fmt + "/", exist_ok=True)  # Use forward slash for consistency

# Save the figure in different formats
for fmt in formats:
    file_path = main_figure_path + fmt + "/" + name + "." + fmt  # Ensure consistent path formatting
    fig.write_image(file_path, format=fmt, engine="kaleido")
    print(f"Saved {fmt.upper()} to: {file_path}")

# Convert PDF to EPS if needed
# call(["pdf2ps", main_figure_path + "pdf" + "/" + "_" + name + ".pdf", main_figure_path + "eps" + "/" + "_" + name + ".eps"])


# summary heatmap

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap, LogNorm
import os

# --- 1. Prepare data ---
daily1 = DF2014.groupby('date')['freq'].sum().reset_index()
daily1['date'] = pd.to_datetime(daily1['date'])
data1 = np.array([daily1['freq']])

daily2 = DF2022.groupby('date')['freq'].sum().reset_index()
daily2['date'] = pd.to_datetime(daily2['date'])
data2 = np.array([daily2['freq']])

# --- 2. Shared log scale ---
epsilon = 1e-3
data1_safe = np.where(data1 <= 0, epsilon, data1)
data2_safe = np.where(data2 <= 0, epsilon, data2)

vmin = min(data1_safe.min(), data2_safe.min())
vmax = max(data1_safe.max(), data2_safe.max())
log_norm = LogNorm(vmin=vmin, vmax=vmax)

# --- 3. Black-and-white colormap ---
custom_cmap = LinearSegmentedColormap.from_list("bw", ["#FFFFFF", "#000000"], N=256)

# --- 4. Figure size in pixels ---
width_px, height_px = 1400, 220
dpi = 100
figsize = (width_px / dpi, height_px / dpi)

# --- 5. Create figure ---
fig, axes = plt.subplots(2, 1, figsize=figsize, dpi=dpi, sharex=True,
                         gridspec_kw={'hspace':0})  # no gap

# --- 6. Heatmaps ---
im1 = axes[0].imshow(data1_safe, aspect='auto', cmap=custom_cmap, norm=log_norm, interpolation='nearest')
axes[0].set_yticks([])
axes[0].set_ylabel('2014', rotation=0, labelpad=25, fontsize=15)

im2 = axes[1].imshow(data2_safe, aspect='auto', cmap=custom_cmap, norm=log_norm, interpolation='nearest')
axes[1].set_yticks([])
axes[1].set_ylabel('2022', rotation=0, labelpad=25, fontsize=15)

# --- 7. X-axis ticks every 7 days ---
dates = daily1['date']
axes[1].set_xticks(np.arange(0, len(dates), 7))
axes[1].set_xticklabels(dates.dt.strftime('%Y-%m-%d').iloc[::7], rotation=90, fontsize=10)
axes[0].set_xticks([])

# --- X-axis ticks every 7 days (weekly) ---
dates = daily1['date']
weekly_idx = np.arange(0, len(dates), 7)
axes[1].set_xticks(weekly_idx)
axes[1].set_xticklabels(dates.dt.strftime('%Y-%m-%d').iloc[::7], rotation=90, fontsize=10)

# Make the weekly ticks a bit longer
axes[1].tick_params(axis='x', which='major', length=8)  # major ticks

# --- 8. Colorbar outside the axes (figure-level) ---
cbar_width = 0.02  # fraction of figure width
cbar_ax = fig.add_axes([0.93, 0.1, cbar_width, 0.8])  # [left, bottom, width, height]
cbar = fig.colorbar(im2, cax=cbar_ax, orientation='vertical')

# Set only main ticks (e.g., powers of 10 for log scale)
main_ticks = [0.001, 0.01]  # adjust depending on your data
cbar.set_ticks(main_ticks)

cbar.set_label('Frequency (log scale)', fontsize=15)
cbar.ax.tick_params(labelsize=13, length=6)  # tick length

# --- 9. Tight layout ---
plt.tight_layout(rect=[0, 0, 0.92, 1])  # leave space for colorbar
plt.show()


In [ ]:
import os
from subprocess import call

# Set the figure directory and filename criteria
sublocation = '/Visualizations/Fig.4_2014vs2022_trends/27022018 and 24022022/2sumfreqbars'

# Define the plot name
name = 'sumfreqs'  # Replace with your desired plot name
#name = 'Wordform_comparison_ADJ'  # Replace with your desired plot name

# Define the base directory for saving files
directory = output_dir
os.chdir(directory)
# Make the main figure directory if not present
main_figure_path = directory + sublocation
os.makedirs(main_figure_path, exist_ok=True)

# Output formats
formats = ['pdf', 'jpeg', 'svg']
for fmt in formats:
    path_fmt = os.path.join(main_figure_path, fmt)
    os.makedirs(path_fmt, exist_ok=True)

    # Save figure
    save_path = os.path.join(path_fmt, f"{name}.{fmt}")
    fig.savefig(save_path, format=fmt, dpi=600 if fmt=='jpeg' else None)
    print(f"Saved {fmt.upper()} to: {save_path}")


# Not used extra

In [ ]:
plot_frequency_comparison(DF2014, DF2022, date_col='date', freq_col='normalized_overal_freq', lang_col='language')


In [ ]:
plot_frequency_comparison(DF2014, DF2022, date_col='date', freq_col='normalized_overal_freq_log', lang_col='language')


# CLUSTERING

In [ ]:
#DF2014 = DF[(DF['date'] >= '2014-01-01') & (DF['date'] <= '2014-04-30')]
#DF2022 = DF[(DF['date'] >= '2022-01-01') & (DF['date'] <= '2022-04-30')]

#DF2014 = DF[(DF['date'] >= '2014-01-30') & (DF['date'] <= '2014-03-27')] #Enne valitud
#DF2022 = DF[(DF['date'] >= '2022-01-27') & (DF['date'] <= '2022-03-24')] # Enne valitud

#DF2014 = DF[(DF['date'] >= '2014-02-06') & (DF['date'] <= '2014-03-19')]
#DF2022 = DF[(DF['date'] >= '2022-02-03') & (DF['date'] <= '2022-03-16')]



# Define the central dates as datetime objects
central_dates = {
    '2014': pd.to_datetime('2014-02-18'),
    '2022': pd.to_datetime('2022-02-24')
}

# Filter the DataFrame based on 4 weeks before and after the central dates
DF2014 = DF[(DF['date'] >= central_dates['2014'] - pd.Timedelta(weeks=1)) & 
            (DF['date'] <= central_dates['2014'] + pd.Timedelta(weeks=3))]

DF2022 = DF[(DF['date'] >= central_dates['2022'] - pd.Timedelta(weeks=4)) & 
            (DF['date'] <= central_dates['2022'] + pd.Timedelta(weeks=4))]

DF2014.head(3)

#DF2014['date'] = DF2014['date'] + pd.DateOffset(years=8, days=6) #offset 2014 to make em comparable

DF2014.head(3)

In [ ]:
#NORMALIZE BY LOCAL AVG
Overal_mean_freq = DF.groupby('language')['freq'].transform('mean')

def calculate_normalized_frequency(df, freq_col='freq', lang_col='language'):
    mean_freq = df.groupby(lang_col)[freq_col].transform('mean')
    normalized_freq = (df[freq_col] / mean_freq)
    return normalized_freq

def calculate_normalized_overal_frequency(df, freq_col='freq', lang_col='language'):
    normalized_freq = np.log10(df[freq_col] / Overal_mean_freq)
    return normalized_freq

# Calculate overall mean frequency for each language in DF
Overal_mean_freq = DF.groupby('language')['freq'].mean()

def calculate_normalized_overal_frequency(df, overall_mean_freq, freq_col='freq', lang_col='language'):
    # Ensure we only calculate for valid entries
    normalized_freq = (df[freq_col] / overall_mean_freq[df[lang_col]])
    return normalized_freq


# Example usage:
DF2014['normalized_loc_freq'] = calculate_normalized_frequency(DF2014)
DF2022['normalized_loc_freq'] = calculate_normalized_frequency(DF2022)
DF2014['normalized_overal_freq'] = calculate_normalized_frequency(DF2014)
DF2022['normalized_overal_freq'] = calculate_normalized_frequency(DF2022)
DF2014['normalized_overal_freq_log'] = np.log10(calculate_normalized_frequency(DF2014))
DF2022['normalized_overal_freq_log'] = np.log10(calculate_normalized_frequency(DF2022))
DF2022['freq_log'] = np.log10(DF2022['freq'])
DF2014['freq_log'] = np.log10(DF2014['freq'])


DF2014.head(3)

In [ ]:
DF2014['normalized_loc_freq'].describe()

In [ ]:
#remove some warnings
import warnings

# Ignore specific KMeans memory leak warnings
warnings.filterwarnings("ignore", category=UserWarning, module='sklearn.cluster._kmeans')

## 2014

In [ ]:
#PLOT CLUSTERS 2014

import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from dtaidistance import dtw

# Reshape data to have each language's time series as a row
df_pivot = DF2014.pivot(index='language', columns='date', values='normalized_loc_freq').fillna(0)

# Convert the DataFrame to a NumPy array for clustering
data_array = df_pivot.values

# Calculate DTW distance matrix
dtw_distance_matrix = dtw.distance_matrix(data_array)

# Define max_clusters
max_clusters = 15  # Set this to your desired maximum number of clusters

# Lists to store metrics
wcss = []  # Within-cluster sum of squares
davies_bouldin_scores = []  # Davies-Bouldin scores

# KMeans clustering and metric calculation
for i in range(1, max_clusters + 1):
    kmeans = KMeans(n_clusters=i, n_init=10, random_state=0)
    
    if i == 1:
        # Fit with only one cluster (will not calculate additional metrics)
        kmeans.fit(df_pivot)
        wcss.append(kmeans.inertia_)
    else:
        # Use DTW distance matrix to predict cluster labels
        cluster_labels = kmeans.fit_predict(dtw_distance_matrix)
        
        # Store metrics
        wcss.append(kmeans.inertia_)
        davies_bouldin_scores.append(davies_bouldin_score(df_pivot, cluster_labels))

# Fill in placeholder values for the first metric scores (not applicable for 1 cluster)
davies_bouldin_scores.insert(0, None)  # Davies-Bouldin is undefined for 1 cluster

# Plotting the metrics
plt.figure(figsize=(10, 5))

# Font size parameters
title_fontsize = 16
label_fontsize = 16
tick_fontsize = 14

# WCSS Plot
plt.subplot(1, 2, 1)
plt.plot(range(1, max_clusters + 1), wcss, marker='o')
plt.title('WCSS', fontsize=title_fontsize)
plt.xlabel('Number of Clusters', fontsize=label_fontsize)
plt.xticks([2, 5, 8, 11, 14], fontsize=tick_fontsize)  # Specified integer ticks for x-axis

# Set y-axis ticks to show 5 points
wcss_min, wcss_max = min(wcss), max(wcss)
plt.yticks(np.linspace(wcss_min, wcss_max, 5), fontsize=tick_fontsize)  # 5 evenly spaced points on y-axis
plt.grid()

# Davies-Bouldin Index Plot
plt.subplot(1, 2, 2)
plt.plot(range(2, max_clusters + 1), davies_bouldin_scores[1:], marker='o')
plt.title('Davies-Bouldin Index', fontsize=title_fontsize)
plt.xlabel('Number of Clusters', fontsize=label_fontsize)
plt.xticks([2, 5, 8, 11, 14], fontsize=tick_fontsize)  # Specified integer ticks for x-axis

# Set y-axis ticks to show 5 points
db_min, db_max = min(davies_bouldin_scores[1:]), max(davies_bouldin_scores[1:])
plt.yticks(np.linspace(db_min, db_max, 5), fontsize=tick_fontsize)  # 5 evenly spaced points on y-axis
plt.grid()

plt.tight_layout()

# Define the output directory and filename
sublocation = '/Visualizations/Clusters/Clustertests/2014_1-3/'  # Your specified subdirectory
base_directory = r'D:\OneDrive\TLU\PhD RESEARCH\Artiklid\Ukraine-Twitter' + sublocation
output_filename = 'Clustertests_2014'  # Desired output file name

# Ensure the base output directory exists
os.makedirs(base_directory, exist_ok=True)

# List of formats to save in, which will also be used as folder names in lowercase
formats = ['pdf', 'jpeg', 'svg']

# Save the figure in each format, with lowercase folder names
for fmt in formats:
    # Create directory for each file type in lowercase
    directory = os.path.join(base_directory, fmt)  # No need to call .lower(), as fmt is already lowercase
    os.makedirs(directory, exist_ok=True)

    # Define output path
    output_path = os.path.join(directory, f"{output_filename}.{fmt}")
    plt.savefig(output_path, format=fmt, dpi=400, bbox_inches='tight')
    print(f"Saved {fmt.upper()} to: {output_path}")

plt.show()


In [ ]:
#GET KMEANS 2014

from sklearn.cluster import KMeans
from dtaidistance import dtw

# Reshape data to have each language's time series as a row
df_pivot = DF2014.pivot(index='language', columns='date', values='normalized_loc_freq').fillna(0)

# Convert the DataFrame to a NumPy array for clustering
data_array = df_pivot.values

# Calculate DTW distance matrix
dtw_distance_matrix = dtw.distance_matrix(data_array)

# Final clustering with the optimal number of clusters
optimal_clusters = 5  # Set this to the optimal number based on previous analysis
kmeans_final = KMeans(n_clusters=optimal_clusters, random_state=0, n_init=10)
final_labels = kmeans_final.fit_predict(dtw_distance_matrix)


In [ ]:
#### RELOAD THE +-4 weeks ####

# Filter the DataFrame based on 4 weeks before and after the central dates
DF2014 = DF[(DF['date'] >= central_dates['2014'] - pd.Timedelta(weeks=4)) & 
            (DF['date'] <= central_dates['2014'] + pd.Timedelta(weeks=4))]
#NORMALIZE BY LOCAL AVG
Overal_mean_freq = DF.groupby('language')['freq'].transform('mean')

def calculate_normalized_frequency(df, freq_col='freq', lang_col='language'):
    mean_freq = df.groupby(lang_col)[freq_col].transform('mean')
    normalized_freq = (df[freq_col] / mean_freq)
    return normalized_freq

def calculate_normalized_overal_frequency(df, freq_col='freq', lang_col='language'):
    normalized_freq = np.log10(df[freq_col] / Overal_mean_freq)
    return normalized_freq

# Calculate overall mean frequency for each language in DF
Overal_mean_freq = DF.groupby('language')['freq'].mean()

def calculate_normalized_overal_frequency(df, overall_mean_freq, freq_col='freq', lang_col='language'):
    # Ensure we only calculate for valid entries
    normalized_freq = (df[freq_col] / overall_mean_freq[df[lang_col]])
    return normalized_freq


# Example usage:
DF2014['normalized_loc_freq'] = calculate_normalized_frequency(DF2014)
DF2014['normalized_overal_freq'] = calculate_normalized_frequency(DF2014)
DF2014['normalized_overal_freq_log'] = np.log10(calculate_normalized_frequency(DF2014))
DF2014['freq_log'] = np.log10(DF2014['freq'])


DF2014.head(3)

In [ ]:
#NEW VERSION CLUSTERS, -1w, +3w

#PLOT CLUSTERED TRENDS 2014

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.dates import DateFormatter
import numpy as np
from sklearn.cluster import KMeans  # Ensure KMeans is imported
import os  # Make sure to import os for directory handling

#### DEFINE DATA
# Reshape data to have each language's time series as a row
df_pivot = DF2014.pivot(index='language', columns='date', values='normalized_loc_freq').fillna(0)
# Apply log transformation to the data, adding 1 to avoid log(0) issues
df_pivotLog = np.log10(df_pivot + 1)

#### CLUSTERS
# Add cluster labels to the DataFrame
df_pivotLog['cluster'] = final_labels
cluster_names = {
    0: 'Medium peaks', 
    1: 'Sharper peaks', 
    2: 'Second peak', 
    3: 'Noisy', 
    4: 'Stable-Noisy', 
} # Specify the desired order of clusters based on unique values
cluster_names = {
    0: 'Medium Double Shock', 
    1: 'Double Shock', 
    2: 'Second Peak Shock', 
    3: 'Noisy Shock', 
    4: 'Weak Noisy Shock', 
} # Specify the desired order of clusters based on unique values

desired_order = [0,3,4,1,2]  # Your specific order
desired_order = [1,0,3,4,2]  # Your specific order

# Calculate consistent y-limits across all clusters
y_min, y_max = df_pivotLog.drop(columns='cluster').min().min(), df_pivotLog.drop(columns='cluster').max().max()

#### PLOT SIZE, LAYOUT
# Define plot layout
fig, axes = plt.subplots(3, 3, figsize=(13, 12))
axes = axes.flatten()  # Flatten axes array for easy indexing

# Define the center date and calculate weekly ticks around it
center_date = pd.to_datetime("2014-02-18")
x_ticks = pd.date_range(start=center_date - pd.Timedelta(weeks=4), 
                        end=center_date + pd.Timedelta(weeks=4), 
                        freq='W-TUE')  # Weekly ticks on Tuesdays

# Iterate over the desired order and create a plot for each in a 3x3 grid
for i, cluster in enumerate(desired_order):
    # Check if the cluster exists in the DataFrame
    if cluster in df_pivotLog['cluster'].unique():
        cluster_data = df_pivotLog[df_pivotLog['cluster'] == cluster]

        # Ensure the columns are in datetime format
        cluster_data.columns = pd.to_datetime(cluster_data.columns, errors='coerce')

        # Plot each language within the current cluster
        ax = axes[i]
        for language in cluster_data.index:
            ax.plot(cluster_data.columns, cluster_data.loc[language], label=language)
        #### TITLES
        # Set title and labels for each subplot using cluster names
        ax.set_title(cluster_names[cluster], fontsize=16)  # Use the cluster name instead of number
        
        #### LEGEND
        # Configure the legend
        ax.legend(loc='upper left', fontsize=14, ncol=2, frameon=False)  # 2 columns and no gray background
        
        #### TICKS
        # Configure the x-axis ticks
        ax.set_xticks(x_ticks)
        ax.set_xticklabels([dt.strftime('%d-%m') for dt in x_ticks], rotation=45, ha="right", fontsize=12)

        ### BOTTOM ROW IS OFF FOR SOME REASON
        # Modify y-axis tick labels to show regular scale (non-logarithmic)
        #y_ticks_log = ax.get_yticks()  # Get current y-ticks (log scale)
        #y_ticks_regular = [10**tick for tick in y_ticks_log]  # Convert them back to the regular scale
        #ax.set_yticklabels([f'{tick:.1f}' for tick in y_ticks_regular], fontsize=14)  # Format labels with one decimal point
        
        # Increase font size for both x and y tick labels
        ax.tick_params(axis='x', labelsize=14)  # Increase x-axis tick label font size
        ax.tick_params(axis='y', labelsize=14)  # Increase y-axis tick label font size
        
        # Set consistent y-limits
        ax.set_ylim(y_min, y_max)

        # Hide y-ticks for all but the leftmost plots
        if i % 3 != 0:  # Check if it's not the first column
            ax.yaxis.set_visible(False)
        
        #BACKGROUND GRID
        # Add vague grid to the x-axis
        ax.xaxis.grid(True, which='major', linestyle='--', linewidth=0.5, alpha=0.5)  # Vague grid for x-axis
        ax.yaxis.grid(False)  # Optional: disable y-axis grid if needed
        
        #BORDERS
        # Remove top and right borders
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)


# Hide any unused subplots if there are fewer than 9 clusters
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

# Adjust layout with reduced gaps
plt.tight_layout()

# Define the output directory and filename
sublocation = '/Visualizations/Clusters/2014/1-3/'  # Your specified subdirectory
base_directory = r'D:\OneDrive\TLU\PhD RESEARCH\Artiklid\Ukraine-Twitter' + sublocation
output_filename = 'Clusters_2014'  # Desired output file name

# Ensure the base output directory exists
os.makedirs(base_directory, exist_ok=True)

# List of formats to save in, which will also be used as folder names in lowercase
formats = ['pdf', 'jpeg', 'svg']

# Save the figure in each format, with lowercase folder names
for fmt in formats:
    # Create directory for each file type in lowercase
    directory = os.path.join(base_directory, fmt)  # No need to call .lower(), as fmt is already lowercase
    os.makedirs(directory, exist_ok=True)

    # Define output path
    output_path = os.path.join(directory, f"{output_filename}.{fmt}")
    plt.savefig(output_path, format=fmt, dpi=400, bbox_inches='tight')
    print(f"Saved {fmt.upper()} to: {output_path}")

# Show the plot
plt.show()


In [ ]:
#OLD CLUSTER VERSION
#PLOT CLUSTERED TRENDS 2014

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.dates import DateFormatter
import numpy as np
from sklearn.cluster import KMeans  # Ensure KMeans is imported
import os  # Make sure to import os for directory handling

# Apply log transformation to the data, adding 1 to avoid log(0) issues
df_pivotLog = np.log10(df_pivot + 1)

# Add cluster labels to the DataFrame
df_pivotLog['cluster'] = final_labels

cluster_names = {
    0: 'Clear peaks 1', 
    1: 'Noisy 3', 
    2: 'Invasion shock', 
    3: 'Clear peaks 2', 
    4: 'Noisy 2', 
    5: 'Noisy 1' ,
    6: 'asd', 
    #7: 'f 2', 
    #8: 'f 1'    
} # Specify the desired order of clusters based on unique values
cluster_names = {
    0: 'Clear peaks 1', 
    1: 'Noisy 3', 
    2: 'Invasion shock', 
    3: 'Clear peaks 2', 
    4: 'Noisy 2', 
    5: 'Noisy 1' ,
    #6: 'asd', 
    #7: 'f 2', 
    #8: 'f 1'    
} # Specify the desired order of clusters based on unique values

#desired_order = [2, 0, 3, 5,4,1]  # Your specific order
#desired_order = [2, 0, 3, 5,4,1, 6]  # Your specific order

# Calculate consistent y-limits across all clusters
y_min, y_max = df_pivotLog.drop(columns='cluster').min().min(), df_pivotLog.drop(columns='cluster').max().max()

# Define plot layout
fig, axes = plt.subplots(3, 3, figsize=(13, 13))
axes = axes.flatten()  # Flatten axes array for easy indexing

# Iterate over the desired order and create a plot for each in a 3x3 grid
for i, cluster in enumerate(desired_order):
    # Check if the cluster exists in the DataFrame
    if cluster in df_pivotLog['cluster'].unique():
        cluster_data = df_pivotLog[df_pivotLog['cluster'] == cluster]

        # Ensure the columns are in datetime format
        cluster_data.columns = pd.to_datetime(cluster_data.columns, errors='coerce')

        # Plot each language within the current cluster
        ax = axes[i]
        for language in cluster_data.index:
            ax.plot(cluster_data.columns, cluster_data.loc[language], label=language)

        # Set title and labels for each subplot using cluster names
        ax.set_title(cluster_names[cluster], fontsize=16)  # Use the cluster name instead of number

        # Configure the legend
        ax.legend(loc='upper left', fontsize=14, ncol=2, frameon=False)  # 2 columns and no gray background

        # Set date format to show only month and day, and increase tick font size
        ax.xaxis.set_major_formatter(DateFormatter('%m-%d'))
        ax.tick_params(axis='both', which='major', labelsize=14)  # Increase tick label font size

        # Set consistent y-limits and rotate x-axis labels
        ax.set_ylim(y_min, y_max)
        plt.setp(ax.get_xticklabels(), rotation=45, ha="right")  # Rotate x-axis labels

        # Hide y-ticks for all but the leftmost plots
        if i % 3 != 0:  # Check if it's not the first column
            ax.yaxis.set_visible(False)

        # Add vague grid to the x-axis
        ax.xaxis.grid(True, which='major', linestyle='--', linewidth=0.5, alpha=0.5)  # Vague grid for x-axis
        ax.yaxis.grid(False)  # Optional: disable y-axis grid if needed

# Hide any unused subplots if there are fewer than 9 clusters
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

# Adjust layout with reduced gaps
plt.tight_layout()

# Define the output directory and filename
sublocation = '/Visualizations/Clusters/alternative_dates_0218_-1w+3w'  # Your specified subdirectory
base_directory = r'D:\OneDrive\TLU\PhD RESEARCH\Artiklid\Ukraine-Twitter' + sublocation
output_filename = 'Clusters_2014_7c'  # Desired output file name

# Ensure the base output directory exists
os.makedirs(base_directory, exist_ok=True)

# List of formats to save in, which will also be used as folder names in lowercase
formats = ['pdf', 'jpeg', 'svg']

# Save the figure in each format, with lowercase folder names
for fmt in formats:
    # Create directory for each file type in lowercase
    directory = os.path.join(base_directory, fmt)  # No need to call .lower(), as fmt is already lowercase
    os.makedirs(directory, exist_ok=True)

    # Define output path
    output_path = os.path.join(directory, f"{output_filename}.{fmt}")
    plt.savefig(output_path, format=fmt, dpi=400, bbox_inches='tight')
    print(f"Saved {fmt.upper()} to: {output_path}")

# Show the plot
plt.show()


## 2022

In [ ]:
#PLOT CLUSTERS 2022

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from dtaidistance import dtw

# Reshape data to have each language's time series as a row
df_pivot = DF2022.pivot(index='language', columns='date', values='normalized_loc_freq').fillna(0)

# Convert the DataFrame to a NumPy array for clustering
data_array = df_pivot.values

# Calculate DTW distance matrix
dtw_distance_matrix = dtw.distance_matrix(data_array)

# Define max_clusters
max_clusters = 15  # Set this to your desired maximum number of clusters

# Lists to store metrics
wcss = []  # Within-cluster sum of squares
davies_bouldin_scores = []  # Davies-Bouldin scores

# KMeans clustering and metric calculation
for i in range(1, max_clusters + 1):
    kmeans = KMeans(n_clusters=i, n_init=10, random_state=0)
    
    if i == 1:
        # Fit with only one cluster (will not calculate additional metrics)
        kmeans.fit(df_pivot)
        wcss.append(kmeans.inertia_)
    else:
        # Use DTW distance matrix to predict cluster labels
        cluster_labels = kmeans.fit_predict(dtw_distance_matrix)
        
        # Store metrics
        wcss.append(kmeans.inertia_)
        davies_bouldin_scores.append(davies_bouldin_score(df_pivot, cluster_labels))

# Fill in placeholder values for the first metric scores (not applicable for 1 cluster)
davies_bouldin_scores.insert(0, None)  # Davies-Bouldin is undefined for 1 cluster

# Plotting the metrics
plt.figure(figsize=(10, 5))

# Font size parameters
title_fontsize = 16
label_fontsize = 16
tick_fontsize = 14

# WCSS Plot
plt.subplot(1, 2, 1)
plt.plot(range(1, max_clusters + 1), wcss, marker='o')
plt.title('WCSS', fontsize=title_fontsize)
plt.xlabel('Number of Clusters', fontsize=label_fontsize)
plt.xticks([2, 5, 8, 11, 14], fontsize=tick_fontsize)  # Setting specified integer ticks
plt.yticks(fontsize=tick_fontsize)
plt.grid()

# Davies-Bouldin Index Plot
plt.subplot(1, 2, 2)
plt.plot(range(2, max_clusters + 1), davies_bouldin_scores[1:], marker='o')
plt.title('Davies-Bouldin Index', fontsize=title_fontsize)
plt.xlabel('Number of Clusters', fontsize=label_fontsize)
plt.xticks([2, 5, 8, 11, 14], fontsize=tick_fontsize)  # Setting specified integer ticks
plt.yticks(fontsize=tick_fontsize)
plt.grid()

plt.tight_layout()

# Define the output directory and filename
sublocation = '/Visualizations/Clusters/Clustertests/2022_4-4'  # Your specified subdirectory
base_directory = r'D:\OneDrive\TLU\PhD RESEARCH\Artiklid\Ukraine-Twitter' + sublocation
output_filename = 'Clustertests_2022'  # Desired output file name

# Ensure the base output directory exists
os.makedirs(base_directory, exist_ok=True)

# List of formats to save in, which will also be used as folder names in lowercase
formats = ['pdf', 'jpeg', 'svg']

# Save the figure in each format, with lowercase folder names
for fmt in formats:
    # Create directory for each file type in lowercase
    directory = os.path.join(base_directory, fmt)  # No need to call .lower(), as fmt is already lowercase
    os.makedirs(directory, exist_ok=True)

    # Define output path
    output_path = os.path.join(directory, f"{output_filename}.{fmt}")
    plt.savefig(output_path, format=fmt, dpi=400, bbox_inches='tight')
    print(f"Saved {fmt.upper()} to: {output_path}")

plt.show()


In [ ]:
#Get KMEANS 2022
from sklearn.cluster import KMeans
from dtaidistance import dtw

# Reshape data to have each language's time series as a row
df_pivot = DF2022.pivot(index='language', columns='date', values='normalized_loc_freq').fillna(0)

# Convert the DataFrame to a NumPy array for clustering
data_array = df_pivot.values

# Calculate DTW distance matrix
dtw_distance_matrix = dtw.distance_matrix(data_array)

# Final clustering with the optimal number of clusters
optimal_clusters = 6  # Set this to the optimal number based on previous analysis
kmeans_final = KMeans(n_clusters=optimal_clusters, random_state=0, n_init=10)
final_labels = kmeans_final.fit_predict(dtw_distance_matrix)


In [ ]:
#### RELOAD THE +-4 weeks ####

# Filter the DataFrame based on 4 weeks before and after the central dates
DF2022 = DF[(DF['date'] >= central_dates['2022'] - pd.Timedelta(weeks=4)) & 
            (DF['date'] <= central_dates['2022'] + pd.Timedelta(weeks=4))]
#NORMALIZE BY LOCAL AVG
Overal_mean_freq = DF.groupby('language')['freq'].transform('mean')

def calculate_normalized_frequency(df, freq_col='freq', lang_col='language'):
    mean_freq = df.groupby(lang_col)[freq_col].transform('mean')
    normalized_freq = (df[freq_col] / mean_freq)
    return normalized_freq

def calculate_normalized_overal_frequency(df, freq_col='freq', lang_col='language'):
    normalized_freq = np.log10(df[freq_col] / Overal_mean_freq)
    return normalized_freq

# Calculate overall mean frequency for each language in DF
Overal_mean_freq = DF.groupby('language')['freq'].mean()

def calculate_normalized_overal_frequency(df, overall_mean_freq, freq_col='freq', lang_col='language'):
    # Ensure we only calculate for valid entries
    normalized_freq = (df[freq_col] / overall_mean_freq[df[lang_col]])
    return normalized_freq


# Example usage:
DF2022['normalized_loc_freq'] = calculate_normalized_frequency(DF2022)
DF2022['normalized_overal_freq'] = calculate_normalized_frequency(DF2022)
DF2022['normalized_overal_freq_log'] = np.log10(calculate_normalized_frequency(DF2022))
DF2022['freq_log'] = np.log10(DF2022['freq'])


In [ ]:
#PLOT CLUSTERED TRENDS 2022
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.dates import DateFormatter
import numpy as np
from sklearn.cluster import KMeans  # Ensure KMeans is imported
import os  # Make sure to import os for directory handling

# Apply log transformation to the data, adding 1 to avoid log(0) issues
df_pivotLog = np.log10(df_pivot + 1)

# Add cluster labels to the DataFrame
df_pivotLog['cluster'] = final_labels

# Define the cluster names
cluster_names = {
    0: 'Shift of state 1',
    1: 'Shock 1',
    2: 'Stable',
    3: 'Shock 2',
    4: 'Shift of state 2',
    5: 'Noisy-mix'
}
cluster_names = {
    0: 'Shock & Sustain 1',
    1: 'Shock & Decay 1',
    2: 'Noisy Sustain',
    3: 'Shock & Decay 2',
    4: 'Shock & Sustain 2',
    5: 'Noisy Shock & Sustain'
}

# Specify the desired order of clusters based on unique values
desired_order = [1, 3, 0, 4, 5, 2]  # Your specific order

# Calculate consistent y-limits across all clusters
y_min, y_max = df_pivotLog.drop(columns='cluster').min().min(), df_pivotLog.drop(columns='cluster').max().max()

# Define plot layout
fig, axes = plt.subplots(3, 3, figsize=(13, 13))
axes = axes.flatten()  # Flatten axes array for easy indexing

# Define the center date and calculate weekly ticks around it
center_date = pd.to_datetime("2022-02-24")
x_ticks = pd.date_range(start=center_date - pd.Timedelta(weeks=4), 
                        end=center_date + pd.Timedelta(weeks=4), 
                        freq='W-THU')  # Weekly ticks on Thursdays

# Iterate over the desired order and create a plot for each in a 3x3 grid
for i, cluster in enumerate(desired_order):
    if cluster in df_pivotLog['cluster'].unique():
        cluster_data = df_pivotLog[df_pivotLog['cluster'] == cluster]
        
        # Make sure the columns are in datetime format
        cluster_data.columns = pd.to_datetime(cluster_data.columns, errors='coerce')

        ax = axes[i]
        for language in cluster_data.index:
            ax.plot(cluster_data.columns, cluster_data.loc[language], label=language)
        
        # Add vertical dotted line (black) on 2022-02-24
        ax.axvline(pd.to_datetime("2022-02-24"), color='black', linestyle=':', linewidth=1)

        # Set subplot title
        ax.set_title(cluster_names[cluster], fontsize=16)
        
        # Configure legend
        ax.legend(loc='upper left', fontsize=14, ncol=2, frameon=False)

        # Set x-axis ticks
        ax.set_xticks(x_ticks)
        ax.set_xticklabels([dt.strftime('%d-%m') for dt in x_ticks],
                           rotation=45, ha="right", fontsize=12)

        # Increase font size for both x and y tick labels
        ax.tick_params(axis='x', labelsize=14)
        ax.tick_params(axis='y', labelsize=14)

        # Set consistent y-limits
        ax.set_ylim(y_min, y_max)

        # Hide y-axis for plots not in the leftmost column
        if i % 3 != 0:
            ax.yaxis.set_visible(False)
        
        # Add faint grid to x-axis
        ax.xaxis.grid(True, which='major', linestyle='--', linewidth=0.5, alpha=0.5)
        ax.yaxis.grid(False)

        # Remove top and right borders
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

# Hide any unused subplots if there are fewer than 9
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()


sublocation = '/Visualizations/Clusters/2022/4-4/'  # Your specified subdirectory
base_directory = r'D:\OneDrive\TLU\PhD RESEARCH\Artiklid\Ukraine-Twitter' + sublocation
output_filename = 'Clusters_2022_with-legend'  # Desired output file name

# Ensure the base output directory exists
os.makedirs(base_directory, exist_ok=True)

# List of formats to save in, which will also be used as folder names in lowercase
formats = ['pdf', 'jpeg', 'svg']

# Save the figure in each format, with lowercase folder names
for fmt in formats:
    # Create directory for each file type in lowercase
    directory = os.path.join(base_directory, fmt)  # No need to call .lower(), as fmt is already lowercase
    os.makedirs(directory, exist_ok=True)

    # Define output path
    output_path = os.path.join(directory, f"{output_filename}.{fmt}")
    plt.savefig(output_path, format=fmt, dpi=400, bbox_inches='tight')
    print(f"Saved {fmt.upper()} to: {output_path}")
    
# Show the plot
plt.show()


## Older

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from dtaidistance import dtw

# Reshape data to have each language's time series as a row
df_pivot = DF2014.pivot(index='language', columns='date', values='normalized_loc_freq').fillna(0)

# Convert the DataFrame to a NumPy array for clustering
data_array = df_pivot.values

# Calculate DTW distance matrix
dtw_distance_matrix = dtw.distance_matrix(data_array)

# Define max_clusters
max_clusters = 15  # Set this to your desired maximum number of clusters

# Lists to store metrics
wcss = []  # Within-cluster sum of squares
silhouette_scores = []  # Silhouette scores
davies_bouldin_scores = []  # Davies-Bouldin scores
calinski_harabasz_scores = []  # Calinski-Harabasz scores

# KMeans clustering and metric calculation
for i in range(1, max_clusters + 1):
    kmeans = KMeans(n_clusters=i, n_init=10, random_state=0)
    
    if i == 1:
        # Fit with only one cluster (will not calculate additional metrics)
        kmeans.fit(df_pivot)
        wcss.append(kmeans.inertia_)
    else:
        # Use DTW distance matrix to predict cluster labels
        cluster_labels = kmeans.fit_predict(dtw_distance_matrix)
        
        # Store metrics
        wcss.append(kmeans.inertia_)
        silhouette_scores.append(silhouette_score(df_pivot, cluster_labels))
        davies_bouldin_scores.append(davies_bouldin_score(df_pivot, cluster_labels))
        calinski_harabasz_scores.append(calinski_harabasz_score(df_pivot, cluster_labels))

# Fill in placeholder values for the first metric scores (not applicable for 1 cluster)
silhouette_scores.insert(0, None)  # Silhouette is undefined for 1 cluster
davies_bouldin_scores.insert(0, None)  # Davies-Bouldin is undefined for 1 cluster
calinski_harabasz_scores.insert(0, None)  # Calinski-Harabasz is undefined for 1 cluster

# Plotting the metrics
plt.figure(figsize=(18, 8))

# Font size parameters
title_fontsize = 20
label_fontsize = 16
tick_fontsize = 14

# WCSS Plot
plt.subplot(2, 2, 1)
plt.plot(range(1, max_clusters + 1), wcss, marker='o')
plt.title('WCSS', fontsize=title_fontsize)
plt.xticks(np.linspace(1, max_clusters, 5), fontsize=tick_fontsize)  # Only 5 ticks
plt.yticks(fontsize=tick_fontsize)
plt.grid()

# Silhouette Score Plot
plt.subplot(2, 2, 2)
plt.plot(range(2, max_clusters + 1), silhouette_scores[1:], marker='o')
plt.title('Silhouette Scores', fontsize=title_fontsize)
plt.xticks(np.linspace(2, max_clusters, 5), fontsize=tick_fontsize)  # Only 5 ticks
plt.yticks(fontsize=tick_fontsize)
plt.grid()

# Davies-Bouldin Index Plot
plt.subplot(2, 2, 3)
plt.plot(range(2, max_clusters + 1), davies_bouldin_scores[1:], marker='o')
plt.title('Davies-Bouldin Index', fontsize=title_fontsize)
plt.xlabel('Number of Clusters', fontsize=label_fontsize)
plt.xticks(np.linspace(2, max_clusters, 5), fontsize=tick_fontsize)  # Only 5 ticks
plt.yticks(fontsize=tick_fontsize)
plt.grid()

# Calinski-Harabasz Index Plot
plt.subplot(2, 2, 4)
plt.plot(range(2, max_clusters + 1), calinski_harabasz_scores[1:], marker='o')
plt.title('Calinski-Harabasz Index', fontsize=title_fontsize)
plt.xlabel('Number of Clusters', fontsize=label_fontsize)
plt.xticks(np.linspace(2, max_clusters, 5), fontsize=tick_fontsize)  # Only 5 ticks
plt.yticks(fontsize=tick_fontsize)
plt.grid()

plt.tight_layout()
plt.show()


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np

def plot_cluster_frequency(df_pivot, cluster_col='cluster'):
    # Get unique clusters
    unique_clusters = df_pivot[cluster_col].unique()

    # Calculate number of rows and columns needed for the grid
    num_clusters = len(unique_clusters)
    num_rows = (num_clusters // 4) + 16
    num_cols = min(num_clusters, 4)

    # Create a grid of subplots with shared x and y axes, and increased vertical spacing
    fig = make_subplots(rows=num_rows, cols=num_cols, subplot_titles=[f'Cluster {cluster}' for cluster in unique_clusters],
                        shared_xaxes=True, shared_yaxes=True, vertical_spacing=0.05, horizontal_spacing=0.02)

    # Iterate through each unique cluster
    for i, cluster in enumerate(unique_clusters):
        cluster_data = df_pivot[df_pivot[cluster_col] == cluster]

        # Ensure that the columns are in datetime format
        cluster_data.columns = pd.to_datetime(cluster_data.columns, errors='coerce')

        # Drop rows where all elements are NaN
        cluster_data = cluster_data.dropna(how='all')

        # Apply transformation: add 1, take log, then subtract 1
        transformed_data = np.log1p(cluster_data)

        # Add a line plot for each language in the current cluster, excluding NaN values
        for language in transformed_data.index:
            # Filter out NaN values for the current language
            lang_data = transformed_data.loc[language].dropna()
            if not lang_data.empty:  # Only plot if there is data
                fig.add_trace(
                    go.Scatter(x=lang_data.index, y=lang_data, mode='lines', name=language, showlegend=False),
                    row=(i // num_cols) + 1, col=(i % num_cols) + 1
                )
    fig.for_each_annotation(lambda a: a.update(font=dict(size=20)))

    # Update layout with reduced margins and increased font size
    fig.update_layout(
        title='',
                title_font=dict(size=20),  # Set title font size to match text size

        margin=dict(l=10, r=10, t=40, b=10),  # Set left, right, top, and bottom margins
        height=800,  # Adjust height as needed
        width=1200,  # Adjust width as needed
        font=dict(size=20)
    )
    
    # Update x and y axes
    fig.update_yaxes(showline=True,
                     linewidth=2, linecolor='black', mirror=True)
    fig.update_xaxes(showline=True, linewidth=1,
                     linecolor='black', mirror=True)

    # Show the plot
    return fig

# Example usage:
fig_clusters = plot_cluster_frequency(df_pivot)
fig_clusters.show()


In [ ]:
!pip install dtaidistance

In [ ]:
import pandas as pd
import numpy as np
import hdbscan
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from dtaidistance import dtw
from dtaidistance import clustering


# Reshape data to have each language's time series as a row
df_pivot = DF2022.pivot(index='language', columns='date', values='freq').fillna(0)

# Convert the DataFrame to a NumPy array for clustering
data_array = df_pivot.values

# Calculate DTW distance matrix
dtw_distance_matrix = dtw.distance_matrix(data_array)
# Perform KMeans clustering with DTW distance
n_clusters = 8  # Define the number of clusters
kmeans = KMeans(n_clusters=n_clusters, random_state=0, n_init=10)
cluster_labels = kmeans.fit_predict(dtw_distance_matrix)

# Add cluster labels to the DataFrame
df_pivot['cluster'] = cluster_labels


# Assuming df_pivot is your DataFrame with the necessary data
for cluster in df_pivot['cluster'].unique():
    cluster_data = df_pivot[df_pivot['cluster'] == cluster]
    
    # Ensure that the columns are in datetime format
    cluster_data.columns = pd.to_datetime(cluster_data.columns, errors='coerce')

    # Now plot
    for language in cluster_data.index:
        plt.plot(cluster_data.columns, cluster_data.loc[language], label=language)

    plt.title(f'Cluster {cluster}')
    plt.xlabel('Date')
    plt.ylabel('Value')  # Change this to your actual y-axis label
    plt.legend()
    plt.show()


In [ ]:
import matplotlib.pyplot as plt

# Number of clusters based on silhouette_scores
num_clusters = range(2, len(silhouette_scores) + 2)  # This will give the correct range for x-axis

# Plot WCSS
plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
plt.plot(range(2, len(wcss) + 2), wcss, marker='o')  # Ensure the range matches wcss length
plt.title('WCSS vs Number of Clusters')
plt.xlabel('Number of Clusters')
plt.ylabel('WCSS')
plt.grid()

# Plot Silhouette Scores
plt.subplot(1, 2, 2)
plt.plot(num_clusters, silhouette_scores, marker='o')
plt.title('Silhouette Scores vs Number of Clusters')
plt.xlabel('Number of Clusters')
plt.ylabel('Silhouette Score')
plt.grid()

plt.tight_layout()
plt.show()


In [ ]:

# Step 2: Reshape data to have each language's time series as a row
df_pivot = DF2022.pivot(index='language', columns='date', values='normalized_loc_freq').fillna(0)

# Step 3: Cluster the languages based on their frequency trends
n_clusters = 8  # Set the number of clusters
kmeans = KMeans(n_clusters=n_clusters, random_state=0)
df_pivot['cluster'] = kmeans.fit_predict(df_pivot)

# Extract only the cluster number and language
df_language_clusters = df_pivot[['cluster']].reset_index().reindex(columns=['cluster', 'language'])

# Step 4: Visualize the clusters
for cluster_num in range(n_clusters):
    cluster_data = df_pivot[df_pivot['cluster'] == cluster_num].drop(columns='cluster').T
    cluster_data.plot(legend=False , title=f"Cluster {cluster_num} Trends")
    plt.xlabel('Date')
    plt.ylabel('Frequency Z-score')
    plt.show()

# Display the DataFrame with cluster number and language
print(df_language_clusters)

# Old stuff

In [ ]:
#COMPARISON 2014 and 2022 NORM
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Assuming 'DFUkraine2022' is your DataFrame containing the dataset
# Get unique languages
unique_languages = DF2022['language'].unique()

DF2014['date'] = DF2014['date'] #offset 2014 to make em comparable

# Calculate number of rows and columns needed for the grid
num_languages = len(unique_languages)
num_rows = (num_languages // 4) + 1
num_cols = min(num_languages, 4)

# Create a 5x5 grid of subplots with shared x and y axes
fig = make_subplots(rows=num_rows, cols=num_cols, subplot_titles=unique_languages, shared_xaxes=True, shared_yaxes=True)

# Define a custom color for all traces
#color = 'orange'

# Iterate through each unique language
for i, language in enumerate(unique_languages):
    # Filter data for the current language
    df_lang = DF2022[DF2022['language'] == language]
    df_lang_2014 = DF2014[DF2014['language'] == language]  # Add this line

    # Calculate subplot row and column
    row = (i // num_cols) + 1
    col = (i % num_cols) + 1
    
    # Add a line plot for 2022
    fig.add_trace(
        go.Scatter(x=df_lang['date'], y=df_lang['normalized_overal_freq'], mode='lines', name=language, line=dict(color='red')),
        row=row, col=col
    )
    
    # Add a line plot for the 2014
    fig.add_trace(
        go.Scatter(x=df_lang_2014['date'], y=df_lang_2014['normalized_overal_freq'], mode='lines', name=language + ' 2014', line=dict(color='blue')),
        row=row, col=col
    )

    
        # Add a vertical line on 24th of February
    fig.add_vline(x='2022-02-24', line=dict(color='black', width=0.9, 
                                            dash='dot'
                                           )
                  , row=row, col=col,
                  #annotation_text='Feb 24', annotation_position='top left'
                 )


# Update layout
fig.update_layout(title='Frequency Change 2014 vs 2022 (1.01-30.04; line corresponds to 27.02.2014 and 24.02.2022)', showlegend=False, height=1200, width=900)

# Set aspect ratio to make subplots square-shaped
fig.update_yaxes(scaleanchor="x", scaleratio=1, type='log',
                showline=True,
                 linewidth=1,
                 linecolor='black',
                 mirror=True)
fig.update_xaxes(constrain='domain',
                 showline=True,
                 linewidth=1,
                 linecolor='black',
                 mirror=True
                    )

# Set custom x-axis ticks

#for i in range(1, num_rows+1):
#    fig.update_yaxes(range=[-2, 3],
#                      ,
#                     tickvals=[0.01, 0.1, 1, 10, 100, 1000, 10000],
#                     row=i, col=1)

fig.update_yaxes(#tickvals=[0.01, 0.1, 1, 10, 100, 1000, 10000],
                 range=[-2, 3],
                #autorange='reversed'
                               )

                 
# Show the plot
fig.show()

In [ ]:

figure='Visualizations\Trend_comparisons'
name='2014-2022_freq_normalized-overal'
directory = r'D:\OneDrive\TLU\PhD RESEARCH\Artiklid\Countries'
os.chdir(directory)
fig.write_image(figure+"/"+name+".pdf", format="pdf", engine="kaleido")
fig.write_image(figure+"/"+name+".png", format="png", engine="kaleido")
from subprocess import call
call(["pdf2ps", figure+"/"+name+".pdf", figure+"/"+name+".eps"])


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Assuming 'DFUkraine2022' is your DataFrame containing the dataset
# Get unique languages
unique_languages = DFUkraine2022['language'].unique()

DFUkraine2014['date'] = DFUkraine2014['date'] #offset 2014 to make them comparable

# Calculate number of rows and columns needed for the grid
num_languages = len(unique_languages)
num_rows = 4
num_cols = 7

# Create a 4x7 grid of subplots with shared x and y axes
fig = make_subplots(rows=num_rows, cols=num_cols, subplot_titles=unique_languages, shared_xaxes=True, shared_yaxes=True)

# Iterate through each unique language
for i, language in enumerate(unique_languages):
    # Filter data for the current language
    df_lang = DFUkraine2022[DFUkraine2022['language'] == language]
    df_lang_2014 = DFUkraine2014[DFUkraine2014['language'] == language]  # Add this line

    # Calculate subplot row and column
    row = (i % num_rows) + 1  # Adjusted row calculation
    col = (i // num_rows) + 1  # Adjusted column calculation
    
    # Add a line plot for 2022
    fig.add_trace(
        go.Scatter(x=df_lang['date'], y=df_lang['freq'], mode='lines', name=language, line=dict(color='red')),
        row=row, col=col
    )
    
    # Add a line plot for the 2014
    fig.add_trace(
        go.Scatter(x=df_lang_2014['date'], y=df_lang_2014['freq'], mode='lines', name=language + ' 2014', line=dict(color='blue')),
        row=row, col=col
    )

    
        # Add a vertical line on 24th of February
    fig.add_vline(x='2022-02-24', line=dict(color='black', width=0.9, 
                                            dash='dot'
                                           )
                  , row=row, col=col,
                  #annotation_text='Feb 24', annotation_position='top left'
                 )


# Update layout
fig.update_layout(
    title='Frequency Change 2014 vs 2022 (1.01-30.04; line corresponds to 27.02.2014 and 24.02.2022)', 
    showlegend=False, 
    height=900, 
    width=1800,
    font=dict(size=22),  # Set main title text size
    margin=dict(l=1, r=1, t=100, b=50),  # Reduce margin between subplots
)

# Set aspect ratio to make subplots square-shaped
fig.update_yaxes(scaleanchor="x", scaleratio=1, type='log',
                showline=True,
                 linewidth=1,
                 linecolor='black',
                 mirror=True)
fig.update_xaxes(constrain='domain',
                 showline=True,
                 linewidth=1,
                 linecolor='black',
                 mirror=True
                    )

# Set subplot titles font size
fig.update_annotations(font=dict(size=28))  # Increase subplot title size

# Set custom x-axis ticks

fig.update_yaxes(#tickvals=[0.01, 0.1, 1, 10, 100, 1000, 10000],
                 tickvals=[0.000001, 0.00001, 0.0001, 0.001, 0.01, 0.1, 1, 10]
                 #range=[0.00001, 0.01],
                #autorange='reversed'
                               )

# Show the plot
fig.show()


In [ ]:

figure='EHAKviz'
name='ComparisonW230224'
directory = r'D:\OneDrive\TLU\PhD RESEARCH\Artiklid\Countries'
os.chdir(directory)
fig.write_image(figure+"/"+name+".pdf", format="pdf", engine="kaleido")
fig.write_image(figure+"/"+name+".png", format="png", engine="kaleido")
from subprocess import call
call(["pdf2ps", figure+"/"+name+".pdf", figure+"/"+name+".eps"])


In [ ]:
#COMPARISON 2014 and 2022 NOT NORMALIZED
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Assuming 'DFUkraine2022' is your DataFrame containing the dataset
# Get unique languages
unique_languages = DF2022['language'].unique()

#DFUkraine2014['date'] = DFUkraine2014['date'] #offset 2014 to make em comparable

# Calculate number of rows and columns needed for the grid
num_languages = len(unique_languages)
num_rows = (num_languages // 4) + 1
num_cols = min(num_languages, 4)

# Create a 5x5 grid of subplots with shared x and y axes
fig = make_subplots(rows=num_rows, cols=num_cols, subplot_titles=unique_languages, shared_xaxes=True, shared_yaxes=True)

# Define a custom color for all traces
#color = 'orange'

# Iterate through each unique language
for i, language in enumerate(unique_languages):
    # Filter data for the current language
    df_lang = DF2022[DF2022['language'] == language]
    df_lang_2014 = DF2014[DF2014['language'] == language]  # Add this line

    # Calculate subplot row and column
    row = (i // num_cols) + 1
    col = (i % num_cols) + 1
    
    # Add a line plot for 2022
    fig.add_trace(
        go.Scatter(x=df_lang['date'], y=df_lang['freq_no_rt'], mode='lines', name=language, line=dict(color='red')),
        row=row, col=col
    )
    
    # Add a line plot for the 2014
    fig.add_trace(
        go.Scatter(x=df_lang_2014['date'], y=df_lang_2014['freq_no_rt'], mode='lines', name=language + ' 2014', line=dict(color='blue')),
        row=row, col=col
    )

    
        # Add a vertical line on 24th of February
    fig.add_vline(x='2022-02-24', line=dict(color='black', width=0.9, 
                                            dash='dot'
                                           )
                  , row=row, col=col,
                  #annotation_text='Feb 24', annotation_position='top left'
                 )


# Update layout
fig.update_layout(title='Frequency Change 2014 vs 2022 (1.01-30.04; line corresponds to 27.02.2014 and 24.02.2022)', showlegend=False, height=1200, width=900)

# Set aspect ratio to make subplots square-shaped
fig.update_yaxes(scaleanchor="x", scaleratio=1, type='log',
                showline=True,
                 linewidth=1,
                 linecolor='black',
                 mirror=True)
fig.update_xaxes(constrain='domain',
                 showline=True,
                 linewidth=1,
                 linecolor='black',
                 mirror=True
                    )

# Set custom x-axis ticks

#for i in range(1, num_rows+1):
#    fig.update_yaxes(range=[-2, 3],
#                      ,
#                     tickvals=[0.01, 0.1, 1, 10, 100, 1000, 10000],
#                     row=i, col=1)

fig.update_yaxes(#tickvals=[0.01, 0.1, 1, 10, 100, 1000, 10000],
                 #range=[-2, 3],
                #autorange='reversed'
                               )

# Show the plot
fig.show()

In [ ]:
#figure='viz/'
figure='Newviz/'
directory = r'D:\OneDrive\TLU\PhD RESEARCH\Artiklid\Countries'
os.chdir(directory)

fig.write_image(figure+"trends28_2014vs2022_NOTNorm.pdf", format="pdf", engine="kaleido")
fig.write_image(figure+"trends28_2014vs2022_NOTNorm.jpeg", format="jpeg", engine="kaleido")
from subprocess import call
call(["pdf2ps", "Figures/"+figure+".pdf", "Figures/"+figure+".eps"])


In [ ]:
DFUkraine2022

In [ ]:
import pandas as pd
from scipy.cluster.hierarchy import linkage, dendrogram
import matplotlib.pyplot as plt

# Extract relevant columns for clustering
df_clustering = DF2014[['date', 'freq', 'language']]
#df_clustering = DFUkraine2022[['date', 'freq', 'language']]

# Pivot the DataFrame to have 'date' as index, 'language' as columns, and 'freq' as values
pivot_df = df_clustering.pivot(index='date', columns='language', values='freq').fillna(0)

# Perform hierarchical clustering on transposed DataFrame
linkage_matrix = linkage(pivot_df.T, method='ward')

# Plot the dendrogram
plt.figure(figsize=(12, 8))
dendrogram(linkage_matrix, labels=pivot_df.columns, leaf_rotation=90)
plt.title('Hierarchical Clustering Dendrogram')
plt.xlabel('Languages')
plt.ylabel('Distance')
plt.show()


In [ ]:
import pandas as pd
from scipy.cluster.hierarchy import linkage, dendrogram
import matplotlib.pyplot as plt

# Extract relevant columns for clustering
#df_clustering = DFUkraine2014[['date', 'freq', 'language']]
df_clustering = DF2022[['date', 'freq', 'language']]

# Pivot the DataFrame to have 'date' as index, 'language' as columns, and 'freq' as values
pivot_df = df_clustering.pivot(index='date', columns='language', values='freq').fillna(0)

# Perform hierarchical clustering on transposed DataFrame
linkage_matrix = linkage(pivot_df.T, method='ward')

# Plot the dendrogram
plt.figure(figsize=(12, 8))
dendrogram(linkage_matrix, labels=pivot_df.columns, leaf_rotation=90)
plt.title('Hierarchical Clustering Dendrogram')
plt.xlabel('Languages')
plt.ylabel('Distance')
plt.show()


In [ ]:
import pandas as pd
from scipy.cluster.hierarchy import linkage, dendrogram
import matplotlib.pyplot as plt

# Extract relevant columns for clustering
#df_clustering = DFUkraine2014[['date', 'freq', 'language']]
df_clustering = DF2014[['date', 'freqNorm', 'language']]

# Pivot the DataFrame to have 'date' as index, 'language' as columns, and 'freq' as values
pivot_df = df_clustering.pivot(index='date', columns='language', values='freqNorm').fillna(0)

# Perform hierarchical clustering on transposed DataFrame
linkage_matrix = linkage(pivot_df.T, method='ward')

# Plot the dendrogram
plt.figure(figsize=(12, 8))
dendrogram(linkage_matrix, labels=pivot_df.columns, leaf_rotation=90)
plt.title('Hierarchical Clustering Dendrogram')
plt.xlabel('Languages')
plt.ylabel('Distance')
plt.show()


In [ ]:
DFUkraine=DFUkraine.reset_index().sort_values(by=['rank'], ascending=True)#[:30] # Järjekorrasta sõnad


In [ ]:
min_ranks

In [ ]:
#PARIM

fig = go.Figure()

fig.add_trace(go.Violin(y=DFUkraine['rank'], x=DFUkraine['language'], line_color='teal',#side='positive',
))
#fig.add_trace(go.Violin(x=CountrFreqDFSmall['count'], y=CountrFreqDFSmall['ngram'], line_color='blue',side='negative',
#))

fig.update_traces(orientation='v', #side='positive',
                  width=5, points=False, spanmode='hard',
                 meanline_visible=True,
                   )

# Find the minimum rank value for each language
min_ranks = DFUkraine.groupby('language')['rank'].min()

# Add labels for minimum ranks above each violin plot
for lang, min_rank in min_ranks.items():
    fig.add_annotation(
        x=lang,
        y=round(np.log10(min_rank), 2)-0.1,  # Convert to log scale
        text=f'{int(min_rank)}',
        #textangle=40,
        showarrow=False,
        #arrowhead=3,
        #ax=0,
    )

fig.update_xaxes(#type="log", #!?!??
                 #range=[1,7000],
                  #autorange='reversed',
                rangemode = "nonnegative",
                showgrid=True,
                zeroline=True,
    tickangle = 45,
    tickson="labels",
    ticks='inside',
    ticklen=5,
      #  categoryorder='category ascending' #Categorize by y axis values - doesnt work!
)

fig.update_yaxes(type="log", #!?!??
                # range=[1,100000],
                  #autorange='reversed',
                rangemode = "nonnegative",
                showgrid=True,
                zeroline=True,
                 autorange='reversed',
                     title = 'Rank',
      # categoryorder='category ascending' #Categorize by y axis values - doesnt work!
)

fig.update_layout(title='Rank distribution of "Ukraine" per language',
    autosize=False,
    width=1200,
    height=500,
                  
)

fig.show()

In [ ]:
figure='viz/'
#fig.write_image(figure+"Violin_28_RankDistribution.pdf", format="pdf", engine="kaleido")
#fig.write_image(figure+"Violin_28_RankDistribution.jpeg", format="jpeg", engine="kaleido")

#call(["pdf2ps", "Figures/"+figure+".pdf", "Figures/"+figure+".eps"])


In [ ]:
DFUkraine=DFUkraine.reset_index().sort_values(by=['count'], ascending=True)#[:30] # Järjekorrasta sõnad


In [ ]:
#PARIM

fig = go.Figure()

fig.add_trace(go.Violin(y=DFUkraine['count'], x=DFUkraine['language'], line_color='teal',#side='positive',
))
#fig.add_trace(go.Violin(x=CountrFreqDFSmall['count'], y=CountrFreqDFSmall['ngram'], line_color='blue',side='negative',
#))

fig.update_traces(orientation='v', #side='positive',
                  width=5, points=False, spanmode='hard',
                 meanline_visible=True,
                   )
fig.update_xaxes(#type="log", #!?!??
                 #range=[1,7000],
                  #autorange='reversed',
                rangemode = "nonnegative",
                showgrid=True,
                zeroline=True,
    tickangle = 45,
    tickson="labels",
    ticks='inside',
    ticklen=5,
      #  categoryorder='category ascending' #Categorize by y axis values - doesnt work!
)

fig.update_yaxes(type="log", #!?!??
                # range=[1,100000],
                  #autorange='reversed',
                rangemode = "nonnegative",
                showgrid=True,
                zeroline=True,
                 autorange='reversed',
                     title = 'Rank',
      # categoryorder='category ascending' #Categorize by y axis values - doesnt work!
)

fig.update_layout(title='Count distribution of "Ukraine" per language',
    autosize=False,
    width=1000,
    height=600,
)

fig.show()

### Ridgeline

In [ ]:

fig = go.Figure()

fig.add_trace(go.Violin(x=DFUkraine['rank'], y=DFUkraine['language'], line_color='teal',
))

fig.update_traces(orientation='h', side='positive', width=10, points=False, spanmode='hard',
                                   meanline_visible=True,

                 )
fig.update_xaxes(#type="log", #!?!??
                 range=[1,1000],
                  #autorange='reversed',
                rangemode = "nonnegative",
                showgrid=True,
                zeroline=True,
    title = 'Rank',
            #categoryorder='category ascending' #doesnt work
)
fig.update_yaxes(#type="log", #!?!??
                 #range=[1,35],
                  #autorange='reversed',
                rangemode = "nonnegative",
                showgrid=True,
                zeroline=True,
    tickson="labels",
    ticks='outside',
    ticklen=5
)

fig.update_layout(title='Top countries 2019.12-2022.12 (median>rank10000)',
    autosize=False,
    width=1000,
    height=2000,
)

fig.show()
